# H infinity condition for stabilization of a discrete time system

In [1]:
import numpy as np
import cvxpy as cp
import control as ct

Matrizes do sistema

In [2]:
A  = np.array([
    [1, 0.035],
    [0, -0.13]
])
Bu = np.array([
    [-0.002],
    [-0.11]
])
Bw = np.array([
    [0.1],
    [0.1]
])
C  = np.array([
    [1, 0]
])
# C = np.eye(A.shape[0])
Du = np.zeros(shape=(C.shape[0], Bu.shape[1]))
Dw = np.zeros(shape=(C.shape[0], Bw.shape[1]))

In [14]:
nx = A.shape[0]
nu = Bu.shape[1]
nc = C.shape[0]
nw = Bw.shape[1]

eps = 10e-19 # 

V = cp.Variable((nx, nx))
Y = cp.Variable((nx, nx), symmetric=True)

constrains = []
constrains += [ Y >> eps ]

B11 = -V - V.T
B12 = V.T@A + Y
B13 = V.T@Bw
B14 = V.T

B22 = -Y
B23 = np.zeros(shape=(nx, nw))
B24 = np.zeros(shape=(nx, nx))

B33 = -np.eye(nw, dtype=float)
B34 = np.zeros(shape=(nw, nx))

B44 = -Y

block = cp.bmat([
    [B11  , B12  , B13  , B14],
    [B12.T, B22  , B23  , B24],
    [B13.T, B23.T, B33  , B34],
    [B14.T, B24.T, B34.T, B44]
])
constrains += [ block << -eps]

prob = cp.Problem(cp.Minimize(None), constraints=constrains)
prob.solve(solver=cp.MOSEK, verbose=True)

                                     CVXPY                                     
                                     v1.5.3                                    
(CVXPY) Dec 17 06:33:33 PM: Your problem has 8 variables, 53 constraints, and 0 parameters.
(CVXPY) Dec 17 06:33:33 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Dec 17 06:33:33 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Dec 17 06:33:33 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Dec 17 06:33:33 PM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Dec 17 06:33:33 PM: Compiling problem (target solver=MOSEK).
(CV

nan

In [18]:
P = np.linalg.inv(Y.value)
P

array([[4.70921668e+08, 1.59801158e+03],
       [1.59801158e+03, 1.63117583e+00]])

In [19]:
all(np.linalg.eig(P).eigenvalues > 0)

True